# Practical No. 02: Text Classification using Embedding Layer and LSTM

**Aim:** Build a text classification model for a binary classification task using an **Embedding layer** and an **LSTM (Long Short-Term Memory) network**, and evaluate its performance using appropriate classification metrics.

**Dataset:** IMDB Movie Reviews Dataset (50,000 movie reviews labeled as **positive** or **negative**), available directly through `tensorflow.keras.datasets`.

---

## Theory / Background

- **Word Embeddings:** Raw text cannot be fed directly into a neural network. Each word in a review is first converted into an integer (token) based on its frequency rank in the vocabulary. The **Embedding layer** then maps each integer/token to a dense, low-dimensional vector that captures semantic meaning — words with similar meaning end up with similar vector representations. Unlike one-hot encoding, embeddings are compact and are *learned* during training.
- **LSTM (Long Short-Term Memory):** A special type of Recurrent Neural Network (RNN) designed to capture long-range dependencies in sequential data (like text) while avoiding the vanishing-gradient problem of vanilla RNNs. It uses gates (input, forget, output) to control what information is remembered or discarded as it reads a sequence word by word.
- **Pipeline used in this notebook:**
  1. Load & explore the IMDB dataset
  2. Preprocess text (tokenize → pad sequences to a fixed length)
  3. Build a model: `Embedding → LSTM → Dense (sigmoid)`
  4. Train the model and monitor accuracy/loss
  5. Evaluate using Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC
  6. Test the model on a few custom sentences


## Step 1: Import Required Libraries

We import TensorFlow/Keras for building the deep learning model, NumPy/Pandas for data handling, Matplotlib/Seaborn for visualization, and scikit-learn for computing classification metrics.

> If a library is missing, uncomment and run the `pip install` line below once.

In [ ]:
# !pip install tensorflow scikit-learn matplotlib seaborn numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix, roc_curve, auc)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## Step 2: Load the Dataset

Keras provides the **IMDB dataset** ready-to-use: 25,000 training and 25,000 testing movie reviews, already converted to sequences of integers (each integer represents a word's rank in the overall frequency of the dataset).

We restrict the vocabulary to the **top 10,000 most frequent words** (`num_words=10000`) — rarer words are dropped, which keeps the Embedding layer small and reduces noise.

In [ ]:
VOCAB_SIZE = 10000   # keep only the top 10,000 most frequent words
MAXLEN = 200          # truncate/pad every review to 200 words

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Training samples:", len(x_train))
print("Testing samples :", len(x_test))
print("Example (encoded) review:\n", x_train[0][:20], "...")
print("Label (0 = negative, 1 = positive):", y_train[0])


## Step 3: Explore the Dataset

Let's decode an encoded review back to readable English (using IMDB's word index), check the class balance, and look at the distribution of review lengths — this justifies our choice of `MAXLEN` for padding.

In [ ]:
# Decode a sample review back into words
word_index = imdb.get_word_index()
# Keras reserves indices 0-3 for padding, start, unknown, unused
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

print("Decoded sample review:\n")
print(decode_review(x_train[0]))
print("\nSentiment label:", "Positive" if y_train[0] == 1 else "Negative")


In [ ]:
# Class balance
unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution in training set:", dict(zip(unique, counts)))

plt.figure(figsize=(5, 4))
sns.countplot(x=y_train)
plt.xticks([0, 1], ["Negative", "Positive"])
plt.title("Class Distribution (Training Set)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()


In [ ]:
# Review length distribution -> helps decide MAXLEN
review_lengths = [len(review) for review in x_train]

plt.figure(figsize=(7, 4))
sns.histplot(review_lengths, bins=50, kde=False)
plt.axvline(MAXLEN, color="red", linestyle="--", label=f"MAXLEN = {MAXLEN}")
plt.title("Distribution of Review Lengths (in words)")
plt.xlabel("Number of words in review")
plt.ylabel("Frequency")
plt.legend()
plt.show()

print("Average review length:", np.mean(review_lengths))
print("Median review length :", np.median(review_lengths))


## Step 4: Preprocess the Text — Padding Sequences

Neural networks require inputs of a **fixed size**, but reviews naturally have different lengths. `pad_sequences` solves this by:
- **Truncating** reviews longer than `MAXLEN` (we keep the last `MAXLEN` words using `truncating='post'` here we truncate from the end).
- **Padding** shorter reviews with zeros so every sequence has exactly `MAXLEN` elements (`padding='post'` adds zeros at the end).

In [ ]:
x_train_pad = pad_sequences(x_train, maxlen=MAXLEN, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test,  maxlen=MAXLEN, padding='post', truncating='post')

print("Shape of padded training data:", x_train_pad.shape)
print("Shape of padded testing data :", x_test_pad.shape)


## Step 5: Build the Model — Embedding + LSTM

Model architecture:

| Layer | Purpose |
|---|---|
| `Input` | Accepts a padded sequence of `MAXLEN` word-indices |
| `Embedding` | Converts each word-index into a dense `EMBED_DIM`-length vector, learned during training |
| `SpatialDropout1D` | Randomly drops entire embedding channels to reduce overfitting |
| `LSTM` | Reads the sequence of word-vectors and captures contextual/sequential patterns |
| `Dense (ReLU)` | A fully-connected layer to learn non-linear combinations of LSTM features |
| `Dropout` | Further regularization before the final decision |
| `Dense (Sigmoid)` | Outputs a probability between 0 and 1 for the **positive** class |

In [ ]:
EMBED_DIM = 32     # size of each word's dense vector
LSTM_UNITS = 64    # number of LSTM memory units

model = Sequential([
    Input(shape=(MAXLEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    SpatialDropout1D(0.2),
    LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')   # binary classification -> single sigmoid unit
])

model.summary()


## Step 6: Compile the Model

- **Loss:** `binary_crossentropy` — standard loss for binary (2-class) classification with a sigmoid output.
- **Optimizer:** `adam` — adaptive learning-rate optimizer, a good default for LSTMs.
- **Metric:** `accuracy` — tracked during training for quick monitoring (full metrics are computed later on the test set).

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


## Step 7: Train the Model

We hold out 20% of the training data as a **validation set** to monitor generalization during training, and use `EarlyStopping` to stop training automatically once validation loss stops improving (prevents overfitting and saves time).

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

EPOCHS = 10
BATCH_SIZE = 128

history = model.fit(
    x_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


## Step 8: Visualize Training History

Plotting accuracy and loss over epochs helps diagnose **overfitting** (training accuracy keeps rising while validation accuracy plateaus/drops) or **underfitting** (both stay low).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='o')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


## Step 9: Evaluate on the Test Set

First, the overall test **loss** and **accuracy** using Keras' built-in `evaluate`.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


## Step 10: Detailed Classification Metrics

Accuracy alone can be misleading (especially on imbalanced data), so we also compute:

- **Precision** — of all reviews predicted positive, how many were actually positive?
- **Recall** — of all actually positive reviews, how many did we correctly find?
- **F1-score** — harmonic mean of precision and recall.
- **Confusion Matrix** — breakdown of correct/incorrect predictions per class.
- **ROC Curve & AUC** — how well the model separates the two classes across all thresholds.

In [ ]:
# Predicted probabilities and hard class labels (threshold = 0.5)
y_pred_prob = model.predict(x_test_pad, verbose=0).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print("\nFull Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# ROC Curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

print(f"AUC Score: {roc_auc:.4f}")


## Step 11: Test the Model on Custom Sentences

Finally, let's sanity-check the model with a few hand-written sentences to see it make real-world predictions.

In [ ]:
import re

def predict_sentiment(text, word_index=word_index, maxlen=MAXLEN, vocab_size=VOCAB_SIZE):
    # Strip punctuation before tokenizing -- otherwise "great!" or "bad," never
    # match the clean word_index entries and silently fall through to <UNK>.
    text = re.sub(r"[^a-z0-9\s]", "", text.lower())
    tokens = text.split()

    encoded = [1]  # <START> token
    for word in tokens:
        rank = word_index.get(word)          # None if the word isn't in the vocab
        if rank is None or rank + 3 >= vocab_size:
            encoded.append(2)                # <UNK> -- do NOT add +3 to this
        else:
            encoded.append(rank + 3)          # keras offsets real ranks by 3

    padded = pad_sequences([encoded], maxlen=maxlen, padding='post', truncating='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Positive" if prob >= 0.5 else "Negative"
    return label, prob

sample_reviews = [
    "This movie was absolutely wonderful, the acting and story were fantastic!",
    "What a waste of time, the plot was boring and the acting was terrible.",
    "The film had brilliant cinematography but a rather slow and dull second half."
]

for review in sample_reviews:
    label, prob = predict_sentiment(review)
    print(f"Review: {review}")
    print(f" -> Predicted Sentiment: {label}  (confidence: {prob:.4f})\n")


## Conclusion

- We built a binary text-classification pipeline using an **Embedding layer** (to learn dense word representations) followed by an **LSTM layer** (to capture sequential/contextual patterns in the review text).
- The model was trained on the IMDB movie review dataset and evaluated using **Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC**, giving a well-rounded picture of its performance beyond a single accuracy number.
- Possible improvements: using pre-trained word embeddings (GloVe/Word2Vec), stacking multiple LSTM/Bidirectional-LSTM layers, tuning hyperparameters (embedding size, LSTM units, dropout), or trying GRU/Transformer-based architectures.
